In [1]:
import pandas as pd
from pathlib import Path
import sys
from IPython.display import display, Markdown

In [2]:
sys.path.append("../src")
from explainer import ForensicExplainer
from models import NetworkRoleClassifier

In [3]:
BASE_DIR = Path.cwd().parent
CHATS_CSV = BASE_DIR / "data" / "processed" / "chats_completos.csv"
NODES_FEATURES_CSV = BASE_DIR / "data" / "processed" / "nodos_features.csv"
ANOMALIES_CSV = BASE_DIR / "data" / "processed" / "chats_top.csv"

In [4]:
explainer = ForensicExplainer(model_name="llama-3.3-70b-versatile")

df_chats = pd.read_csv(CHATS_CSV)
df_features = pd.read_csv(NODES_FEATURES_CSV)
df_anomalies = pd.read_csv(ANOMALIES_CSV)

In [5]:
clasificador = NetworkRoleClassifier(n_clusters=4)
features = ['In_Degree', 'Out_Degree', 'Betweenness', 'PageRank', 'Hub_Score', 'Authority_Score']
df_model, _, _ = clasificador.fit_predict(df_features, features)

In [6]:
# SECCIÓN 1: EXPLICABILIDAD DE GRAFOS
display(Markdown("## 🧠 1. Análisis Estructural y Perfilado de Actores Clave"))
display(Markdown("Evaluación semántica de los 5 nodos con mayor poder jerárquico (PageRank) en la red."))

# Filtramos los 5 actores principales
top_actores = df_model.nlargest(5, 'PageRank')

for _, row in top_actores.iterrows():
    actor = row['Nodo']
    rol = row.get('Etiqueta_Forense', 'Desconocido')
    pr_score = row['PageRank']
    
    # Filtramos mensajes del actor
    df_actor = df_chats[df_chats['Emisor'].str.upper().str.contains(str(actor).upper(), na=False)]
    
    mensajes_muestra = []
    # Ordenamos por longitud para coger los más sustanciales
    for _, msg_row in df_actor.sort_values(by='Mensaje', key=lambda x: x.str.len(), ascending=False).head(5).iterrows():
        # Formateamos incluyendo la emoción
        mensajes_muestra.append(f"[Emoción: {msg_row['NLP_Emocion']}] - {msg_row['Mensaje']}")
        
    if mensajes_muestra:
        explicacion = explainer.explain_actor_profile(actor, rol, pr_score, "\n".join(mensajes_muestra))
        display(Markdown(f"### 👤 Actor: `{actor}`"))
        display(Markdown(f"- **Jerarquía Matemática:** `{rol}` (PageRank: {pr_score:.4f})"))
        display(Markdown(f"- **Evaluación Forense (LLM + NLP):**\n> {explicacion}\n"))
        display(Markdown("---"))

## 🧠 1. Análisis Estructural y Perfilado de Actores Clave

Evaluación semántica de los 5 nodos con mayor poder jerárquico (PageRank) en la red.

### 👤 Actor: `RODOLFO REYES`

- **Jerarquía Matemática:** `Cúpula Directiva (Líderes)` (PageRank: 0.0914)

- **Evaluación Forense (LLM + NLP):**
> Rodolfo REYES muestra un patrón de comportamiento con un "Afecto Plano" debido a la predominancia de emociones 'OTHERS' en sus interacciones, lo que sugiere un tono corporativo y controlado. Su rol como líder en la "Cúpula Directiva" se valida a través de su participación en conversaciones estratégicas y su conexión con una red de influencia. Su comunicación refleja un enfoque en la resolución de asuntos y la búsqueda de soluciones, sin dejar traslucir emociones personales. Esto es consistente con el comportamiento esperado de un operador de alto nivel en gestiones de lobby o influencia.


...

---

### 👤 Actor: `JULIO MARTINEZ MARTINEZ`

- **Jerarquía Matemática:** `Operadores Periféricos / Ejecutores` (PageRank: 0.0238)

- **Evaluación Forense (LLM + NLP):**
> La evaluación forense sugiere que 'JULIO MARTINEZ MARTINEZ' desempeña un rol de 'Operadores Periféricos / Ejecutores' con un nivel de jerarquía moderado. El predominio de emociones 'OTHERS' o Neutras en sus comunicaciones indica un 'Afecto Plano', característico de operadores de alto nivel que utilizan un tono corporativo para evitar dejar evidencia emocional. Esto valida su clasificación inicial y sugiere un posible involucramiento en gestiones de lobby o influencia. Su comportamiento lingüístico respalda su rol asignado.


---

In [7]:
display(df_chats)

,Pagina_PDF,Formato,Fecha,Hora,Emisor,Mensaje,NLP_Sentimiento,NLP_Emocion
0,30,RegEx_F1,23/3/20,23:06:48,Miguel Palomero,Que vais a hacer con la línea aérea?,NEG,others
1,30,RegEx_F1,23/3/20,23:07:01,Miguel Palomero,"Has visto BA, AE y Iberia?",NEU,others
2,30,RegEx_F1,23/3/20,23:07:04,Rodolfo Reyes,Aguantando la paliza,NEU,others
3,30,RegEx_F1,23/3/20,23:07:16,Miguel Palomero,Me imagino,NEU,others
4,30,RegEx_F1,23/3/20,23:07:41,Rodolfo Reyes,Necesitamos llegar a las ayudas,NEU,others
...,...,...,...,...,...,...,...,...
825,153,RegEx_F2,17/08/2021,11:27:07,JULIO MARTINEZ MARTINEZ,dale duro,NEU,others
826,153,RegEx_F2,17/08/2021,12:32:16,JULIO MARTINEZ MARTINEZ,pone plus elmundo nombrado sepi polemica es em...,NEU,others
827,153,RegEx_F2,17/08/2021,12:33:17,RODOLFO REYES,conoces lo,NEU,others
828,153,RegEx_F2,17/08/2021,12:48:17,RODOLFO REYES,10k done,NEU,joy


In [8]:
# 1. Forzamos la conversión. Cualquier texto no válido (o nulo) se transformará en NaT
df_chats['Datetime'] = pd.to_datetime(
    df_chats['Fecha'] + ' ' + df_chats['Hora'], 
    format='mixed', 
    dayfirst=True, 
    errors='coerce'
)

# 2. Creamos una máscara booleana para identificar los registros que eran inválidos
is_na = df_chats['Datetime'].isna()

# 3. Calculamos cuántos NaT consecutivos hay en cada bloque inválido
# (~is_na).cumsum() genera un identificador de grupo que se mantiene igual para cada racha de nulos
consecutive_nas = is_na.groupby((~is_na).cumsum()).cumsum()

# 4. Propagamos el último datetime válido hacia adelante (forward fill)
base_datetime = df_chats['Datetime'].ffill()

# Caso extremo: Si el DataFrame arranca con filas inválidas, no habrá "registro anterior".
# Como solución, tomamos la Fecha de esa fila y le asignamos las 00:00:00 de base.
fecha_base_inicio = pd.to_datetime(df_chats['Fecha'] + ' 00:00:00', format='mixed', dayfirst=True, errors='coerce')
base_datetime = base_datetime.fillna(fecha_base_inicio)

# 5. Sumamos los minutos correspondientes (0 min si era válido, 1 min al primero inválido, 2 min al segundo, etc.)
df_chats['Datetime'] = base_datetime + pd.to_timedelta(consecutive_nas, unit='min')

In [9]:
# SECCIÓN 2: EXPLICABILIDAD DE ANOMALÍAS TEMPORALES
display(Markdown("---"))
display(Markdown("## 🚨 2. Análisis Semántico de Anomalías de Comportamiento"))
display(Markdown("Evaluación de los 5 eventos temporales más atípicos detectados por el algoritmo Isolation Forest."))

# Fix Datetime
df_anomalies['Datetime'] = pd.to_datetime(df_anomalies['Datetime'])

# Ordenamos por Anomaly_Score (los más negativos son los más anómalos)
df_anom_enrich = pd.merge(df_anomalies, df_chats[['Datetime', 'Mensaje', 'NLP_Emocion']], on=['Datetime', 'Mensaje'], how='left')
top_anomalias = df_anom_enrich.sort_values(by='Anomaly_Score', ascending=True).head(5)

for i, row in top_anomalias.iterrows():
    fecha = str(row['Datetime'])
    hora = f"{row['Hour']}:00"
    texto = str(row['Mensaje'])
    longitud = row['Message_Length']
    score = row['Anomaly_Score']
    emocion = str(row.get('NLP_Emocion', 'Neutro'))
    
    explicacion = explainer.explain_anomaly(fecha, hora, longitud, score, emocion, texto)
    
    display(Markdown(f"### ⚠️ Anomalía Crítica Nivel {i+1}"))
    display(Markdown(f"- **Metadatos:** `Hora: {hora} | Score: {score:.3f} | Emoción NLP: {emocion.upper()}`"))
    display(Markdown(f"- **Texto Base:** _{texto[:150]}..._"))
    display(Markdown(f"- **Evaluación Forense (LLM + NLP):**\n> {explicacion}\n"))
    display(Markdown("---"))

---

## 🚨 2. Análisis Semántico de Anomalías de Comportamiento

Evaluación de los 5 eventos temporales más atípicos detectados por el algoritmo Isolation Forest.

### ⚠️ Anomalía Crítica Nivel 1

- **Metadatos:** `Hora: 9:00 | Score: -0.056 | Emoción NLP: OTHERS`

...

### ⚠️ Anomalía Crítica Nivel 5

- **Metadatos:** `Hora: 1:00 | Score: -0.034 | Emoción NLP: OTHERS`

- **Texto Base:** _Moncho Gordils por aqui..._

- **Evaluación Forense (LLM + NLP):**
> El mensaje "Moncho Gordils por aqui" presenta una anomalía matemática crítica con un score de -0.034 y una emoción neutra ("OTHERS"), lo que sugiere una posible coordinación clandestina o lenguaje codificado. La longitud del mensaje y la hora atípica de envío refuerzan esta hipótesis. Esto puede indicar una gestión crítica o presión para realizar una acción específica de manera encubierta. La neutralidad emocional puede ser una táctica para evitar detectar intenciones reales.


---